<a href="https://colab.research.google.com/github/Husseinhhameed/Mobile-CoHAtNet/blob/main/estimate_IMU_from_Transformation_matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from scipy.interpolate import CubicSpline
from google.colab import drive

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Function to load transformation matrices from text files
def load_transformation_matrices(folder_path):
    transformation_matrices = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith('.pose.txt'):
            file_path = os.path.join(folder_path, filename)
            matrix = np.loadtxt(file_path)
            transformation_matrices.append(matrix)
    return transformation_matrices

In [ ]:
# Function to decompose transformation matrix into rotation and translation
def decompose_transformation_matrix(matrix):
    rotation = matrix[:3, :3]
    translation = matrix[:3, 3]
    return rotation, translation

In [ ]:
# Function to interpolate poses using cubic splines
def interpolate_trajectory(timestamps, translations, rotations):
    translations = np.array(translations)
    rotations = np.array([R.from_matrix(rot).as_quat() for rot in rotations])  # Convert to quaternions

    translation_splines = [CubicSpline(timestamps, translations[:, i]) for i in range(3)]
    rotation_splines = [CubicSpline(timestamps, rotations[:, i]) for i in range(4)]  # Quaternions have 4 components

    return translation_splines, rotation_splines

In [ ]:
# Function to compute angular velocity from quaternions
def compute_angular_velocity(quaternions, timestamps):
    angular_velocities = []
    for i in range(len(timestamps) - 1):
        q1 = R.from_quat(quaternions[i])
        q2 = R.from_quat(quaternions[i + 1])
        delta_t = timestamps[i + 1] - timestamps[i]
        relative_rotation = q2 * q1.inv()
        angular_velocity = relative_rotation.as_rotvec() / delta_t  # Convert to angular velocity
        angular_velocities.append(angular_velocity)
    return np.array(angular_velocities)

In [ ]:
import numpy as np
# Function to compute linear acceleration
def compute_linear_acceleration(translations, timestamps, gravity=np.array([0, 0, -9.81])):
    linear_accelerations = []
    for i in range(1, len(timestamps) - 1):
        delta_t1 = timestamps[i] - timestamps[i - 1]
        delta_t2 = timestamps[i + 1] - timestamps[i]
        accel = (translations[i + 1] - 2 * translations[i] + translations[i - 1]) / (delta_t1 * delta_t2)
        accel += gravity  # Adjust for gravity
        linear_accelerations.append(accel)
    return np.array(linear_accelerations)

In [ ]:
# Function to add noise and biases to IMU data
def add_noise_and_bias(data, noise_std=0.01, bias=0.001):
    noise = np.random.normal(0, noise_std, data.shape)
    bias_vector = np.ones(data.shape) * bias
    return data + noise + bias_vector

In [ ]:
import os
from scipy.spatial.transform import Rotation as R
import random


In [ ]:
# Main function
if __name__ == "__main__":
    # Path to the folder containing the sequences
    office_scene_path = "/content/drive/MyDrive/7-Scense/office/seq-01"

    # Load transformation matrices
    transformation_matrices = load_transformation_matrices(office_scene_path)

    # Extract timestamps, translations, and rotations
    timestamps = np.arange(len(transformation_matrices))  # Assuming one frame per unit time
    translations = []
    rotations = []

    for matrix in transformation_matrices:
        rotation, translation = decompose_transformation_matrix(matrix)
        translations.append(translation)
        rotations.append(rotation)

    # Interpolate trajectories
    translation_splines, rotation_splines = interpolate_trajectory(timestamps, translations, rotations)

    # Generate data for random 10 frames
    random_frames = random.sample(range(len(timestamps)), 10)
    random_frames.sort()

    selected_timestamps = timestamps[random_frames]

    # Interpolate translations and rotations for the selected timestamps
    interpolated_translations = np.array([
        [spline(ts) for spline in translation_splines] for ts in selected_timestamps
    ])
    interpolated_rotations = np.array([
        [spline(ts) for spline in rotation_splines] for ts in selected_timestamps
    ])

    # Ensure interpolated_rotations are converted into quaternions properly
    interpolated_rotations = np.array(interpolated_rotations).reshape(-1, 4)

    # Compute angular velocities and linear accelerations
    angular_velocities = compute_angular_velocity(interpolated_rotations, selected_timestamps)
    linear_accelerations = compute_linear_acceleration(interpolated_translations, selected_timestamps)


    # Add noise and biases
    angular_velocities_noisy = add_noise_and_bias(angular_velocities)
    linear_accelerations_noisy = add_noise_and_bias(linear_accelerations)

    # Print the results
    for i in range(len(angular_velocities_noisy)):
        print(f"Frame {random_frames[i]}:")
        print(f"  Angular Velocity (Noisy): {angular_velocities_noisy[i]}")
        if i < len(linear_accelerations_noisy):
            print(f"  Linear Acceleration (Noisy): {linear_accelerations_noisy[i]}")



Frame 19:
  Angular Velocity (Noisy): [ 0.0191263  -0.00726504  0.01015059]
  Linear Acceleration (Noisy): [-6.21815056e-03  6.20409428e-04 -9.81867621e+00]
Frame 157:
  Angular Velocity (Noisy): [-0.01759447  0.00272813  0.00756003]
  Linear Acceleration (Noisy): [ 7.07328351e-03  7.74250158e-03 -9.79162149e+00]
Frame 231:
  Angular Velocity (Noisy): [ 0.00177522  0.01612613 -0.00453577]
  Linear Acceleration (Noisy): [-0.01860562  0.0150598  -9.82268229]
Frame 339:
  Angular Velocity (Noisy): [0.01261637 0.01556786 0.01636889]
  Linear Acceleration (Noisy): [ 8.04493454e-03  5.84471936e-03 -9.80721375e+00]
Frame 417:
  Angular Velocity (Noisy): [-0.00242186  0.00757019  0.00859466]
  Linear Acceleration (Noisy): [ 1.51180006e-02 -6.26039662e-03 -9.78269041e+00]
Frame 439:
  Angular Velocity (Noisy): [-0.00924082  0.01372993  0.00371171]
  Linear Acceleration (Noisy): [-4.70519053e-04  4.70139186e-03 -9.81891726e+00]
Frame 511:
  Angular Velocity (Noisy): [-0.01199932  0.00535775  0.0

# The Follwing Script is calculate and save imu for specific folder within dataset

In [ ]:
# Mount Google Drive (Colab only)
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from scipy.interpolate import CubicSpline
from google.colab import drive
import os
import numpy as np
from scipy.spatial.transform import Rotation as R
import random

In [ ]:
def load_transformation_matrices(folder_path):
    """
    Loads all 4x4 transformation matrices from text files ending with `.pose.txt`.
    Returns a list of matrices and a list of filenames.
    """
    transformation_matrices = []
    file_names = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith('.pose.txt'):
            file_path = os.path.join(folder_path, filename)
            matrix = np.loadtxt(file_path)
            transformation_matrices.append(matrix)
            file_names.append(filename)
    return transformation_matrices, file_names


In [ ]:
def decompose_transformation_matrix(matrix):
    """
    Decompose a 4x4 matrix into a 3x3 rotation and a 3D translation vector.
    """
    rotation = matrix[:3, :3]
    translation = matrix[:3, 3]
    return rotation, translation

In [ ]:
def interpolate_trajectory(timestamps, translations, rotations):
    """
    Interpolate translations/rotations over time using cubic splines.

    - translations: Nx3
    - rotations: Nx3x3, converted internally to quaternions => Nx4
    - Returns: lists of splines (translation_splines, rotation_splines)
    """
    # Ensure array shapes
    translations = np.array(translations)  # shape (N, 3)
    # Convert each rotation matrix to quaternion
    rotations_quat = np.array([R.from_matrix(rot).as_quat() for rot in rotations])  # (N, 4)

    # Build cubic splines for x, y, z
    translation_splines = [CubicSpline(timestamps, translations[:, i]) for i in range(3)]
    # Build cubic splines for quaternion components (w, x, y, z)
    # NOTE: scipy's as_quat returns [x, y, z, w] order, but let's just keep it consistent.
    rotation_splines = [CubicSpline(timestamps, rotations_quat[:, i]) for i in range(4)]

    return translation_splines, rotation_splines

In [ ]:
def compute_angular_velocity(quaternions, timestamps):
    """
    EXACT code from your snippet:
    Computes angular velocity from consecutive quaternions.
    - quaternions: shape (N, 4) [in some order, e.g., x,y,z,w]
    - timestamps: shape (N,)
    Returns (N-1, 3) angular velocities.
    """
    angular_velocities = []
    for i in range(len(timestamps) - 1):
        q1 = R.from_quat(quaternions[i])
        q2 = R.from_quat(quaternions[i + 1])
        delta_t = timestamps[i + 1] - timestamps[i]
        relative_rotation = q2 * q1.inv()
        # Convert to axis-angle => rotvec
        angular_velocity = relative_rotation.as_rotvec() / delta_t
        angular_velocities.append(angular_velocity)
    return np.array(angular_velocities)

In [ ]:
def compute_linear_acceleration(translations, timestamps, gravity=np.array([0, 0, -9.81])):
    """
    EXACT code from your snippet:
    Computes linear acceleration via discrete second derivative + gravity.
    - translations: shape (N, 3)
    - timestamps: shape (N,)
    Returns (N-2, 3) linear accelerations.
    """
    linear_accelerations = []
    for i in range(1, len(timestamps) - 1):
        delta_t1 = timestamps[i] - timestamps[i - 1]
        delta_t2 = timestamps[i + 1] - timestamps[i]
        accel = (translations[i + 1] - 2 * translations[i] + translations[i - 1]) / (delta_t1 * delta_t2)
        accel += gravity
        linear_accelerations.append(accel)
    return np.array(linear_accelerations)

In [ ]:
def add_noise_and_bias(data, noise_std=0.01, bias=0.001):
    """
    EXACT code from your snippet:
    Adds Gaussian noise and constant bias to data.
    """
    noise = np.random.normal(0, noise_std, data.shape)
    bias_vector = np.ones(data.shape) * bias
    return data + noise + bias_vector

In [ ]:
def simulate_imu_for_folder(folder_path):
    """
    1) Load all 4x4 poses from `folder_path` (`*.pose.txt`)
    2) Build timestamps (0,1,2,...)
    3) Decompose each pose -> translation & rotation
    4) Interpolate the entire trajectory if desired (or keep it discrete)
    5) Compute angular velocity and linear acceleration for all frames
    6) Add noise & bias
    7) Write each frame's IMU data to a .imu.txt with the same base name.
    """
    transformation_matrices, file_names = load_transformation_matrices(folder_path)
    if not transformation_matrices:
        print(f"No pose files found in {folder_path}")
        return

    # 1) Create timestamps (assuming uniform 1 unit per frame)
    N = len(transformation_matrices)
    timestamps = np.arange(N)

    # 2) Extract raw translations & rotations
    translations = []
    rotations = []
    for mat in transformation_matrices:
        rot, trans = decompose_transformation_matrix(mat)
        translations.append(trans)
        rotations.append(rot)

    translations = np.array(translations)  # shape (N, 3)
    rotations = np.array(rotations)        # shape (N, 3, 3)

    # 3) Interpolate if you want smooth data, or skip if you want discrete
    #    We'll keep the same approach as your code: cubic spline -> then sample at each integer time
    translation_splines, rotation_splines = interpolate_trajectory(timestamps, translations, rotations)

    # 4) Evaluate the splines at each integer timestamp
    interpolated_translations = np.array([
        [spline(t) for spline in translation_splines] for t in timestamps
    ])  # shape (N, 3)
    interpolated_rotations = np.array([
        [spline(t) for spline in rotation_splines] for t in timestamps
    ])  # shape (N, 4) [x,y,z,w per row]

    # 5) Now compute angular velocity & linear acceleration with your exact functions
    #    Note these produce arrays that are (N-1) and (N-2) in length, so we'll handle edges carefully.
    angular_vels = compute_angular_velocity(interpolated_rotations, timestamps)       # shape (N-1, 3)
    linear_accs = compute_linear_acceleration(interpolated_translations, timestamps)  # shape (N-2, 3)

    # 6) Add noise & bias
    angular_vels_noisy = add_noise_and_bias(angular_vels)
    linear_accs_noisy = add_noise_and_bias(linear_accs)

    # 7) Write out for each frame in the dataset
    #    - We'll produce exactly 1 line in the .imu.txt for each .pose.txt
    #    - The "missing" data for the first/last frames are set to zero.
    #
    #    So, frame i has:
    #       angular velocity = angular_vels_noisy[i] if i < N-1 else 0
    #       linear accel     = linear_accs_noisy[i-1] if 0 < i < N-1 else 0
    #
    #    Then we write them to the file with same base name as pose file.

    for i in range(N):
        pose_file_name = file_names[i]                               # e.g. "000000.pose.txt"
        imu_file_name = pose_file_name.replace(".pose.txt", ".imu.txt")
        imu_file_path = os.path.join(folder_path, imu_file_name)

        # Angular velocity
        if i < len(angular_vels_noisy):
            av = angular_vels_noisy[i]
        else:
            av = np.array([0.0, 0.0, 0.0])

        # Linear acceleration
        # The center difference is valid from i=1..N-2 => index i-1 in linear_accs_noisy
        if 1 <= i <= (N-2):
            la = linear_accs_noisy[i-1]
        else:
            la = np.array([0.0, 0.0, 0.0])

        # Write 1 line: avX avY avZ laX laY laZ
        with open(imu_file_path, 'w') as f:
            f.write(f"{av[0]:.6f} {av[1]:.6f} {av[2]:.6f} "
                    f"{la[0]:.6f} {la[1]:.6f} {la[2]:.6f}\n")

    print(f"Saved IMU data for folder: {folder_path}")

In [ ]:
if __name__ == "__main__":
    # Path to your main office dataset
    office_scene_path = "/content/drive/MyDrive/7-Scense/office"

    # Suppose you have subfolders: seq-01, seq-02, ..., seq-10
    subfolders = [f"seq-{i:02d}" for i in range(1, 11)]

    for seq_name in subfolders:
        folder_path = os.path.join(office_scene_path, seq_name)
        if os.path.isdir(folder_path):
            simulate_imu_for_folder(folder_path)
        else:
            print(f"Skipping {folder_path}: not found or not a directory.")

Saved IMU data for folder: /content/drive/MyDrive/7-Scense/office/seq-01
Saved IMU data for folder: /content/drive/MyDrive/7-Scense/office/seq-02
Saved IMU data for folder: /content/drive/MyDrive/7-Scense/office/seq-03
Saved IMU data for folder: /content/drive/MyDrive/7-Scense/office/seq-04
Saved IMU data for folder: /content/drive/MyDrive/7-Scense/office/seq-05
Saved IMU data for folder: /content/drive/MyDrive/7-Scense/office/seq-06
Saved IMU data for folder: /content/drive/MyDrive/7-Scense/office/seq-07
Saved IMU data for folder: /content/drive/MyDrive/7-Scense/office/seq-08
Saved IMU data for folder: /content/drive/MyDrive/7-Scense/office/seq-09
Saved IMU data for folder: /content/drive/MyDrive/7-Scense/office/seq-10
